# 캐글 대회

In [ ]:
# 필요한 모듈 임포트
import os, hds
import pandas as pd
import numpy as np
from plt_rcs import *

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import rankdata

In [23]:
os.getcwd()

'c:\\Users\\pc\\Desktop\\Repo\\SeSAC\\study\\sesac_ml_dl_study_repo\\kaggle_data_2\\data'

In [24]:
os.chdir('../data')

In [25]:
sorted(os.listdir())

['sample_submission.csv', 'test.csv', 'train.csv']

In [26]:
# 데이터셋 불러오기
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

## 데이터 조회

In [27]:
train.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Num ID
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,14860
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,47181
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,47182
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,47183
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,47184


In [28]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Target                   10000 non-null  int64  
 9   Failure Type             10000 non-null  object 
 10  Num ID                   10000 non-null  int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB


In [29]:
train.describe().round(3)

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Num ID
count,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000
mean,5000.500,300.005,310.006,1538.776,39.987,107.951,0.034,40711.266
std,2886.896,2.000,1.484,179.284,9.969,63.654,0.181,14870.161
min,1.000,295.300,305.700,1168.000,3.800,0.000,0.000,14860.000
25%,2500.750,298.300,308.800,1423.000,33.200,53.000,0.000,23214.750
50%,5000.500,300.100,310.100,1503.000,40.100,108.000,0.000,48861.500
75%,7500.250,301.500,311.100,1612.000,46.800,162.000,0.000,53001.500
max,10000.000,304.500,313.800,2886.000,76.600,253.000,1.000,57174.000


In [30]:
train.describe(include=object)

,Product ID,Type,Failure Type
count,10000,10000,10000
unique,10000,3,6
top,M14860,L,No Failure
freq,1,6000,9652


In [31]:
train['Failure Type'].value_counts()

Failure Type
No Failure                  9652
Heat Dissipation Failure     112
Power Failure                 95
Overstrain Failure            78
Tool Wear Failure             45
Random Failures               18
Name: count, dtype: int64

In [32]:
test.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type
0,10001,M24860,M,300.5,309.5,1451,47.8,93,No Failure
1,10002,M24861,M,301.4,311.1,1697,35.6,160,No Failure
2,10003,H39413,H,304.0,313.1,1612,33.7,100,No Failure
3,10004,L57175,L,298.6,310.5,1276,55.2,91,No Failure
4,10005,L57176,L,299.9,310.8,1400,46.2,219,No Failure


In [33]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      3000 non-null   int64  
 1   Product ID               3000 non-null   object 
 2   Type                     3000 non-null   object 
 3   Air temperature [K]      3000 non-null   float64
 4   Process temperature [K]  3000 non-null   float64
 5   Rotational speed [rpm]   3000 non-null   int64  
 6   Torque [Nm]              3000 non-null   float64
 7   Tool wear [min]          3000 non-null   int64  
 8   Failure Type             3000 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 211.1+ KB


## 특성 행렬과 타겟 벡터로 분리

In [34]:
train.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Target', 'Failure Type', 'Num ID'],
      dtype='object')

In [35]:
# 타겟 벡터명 설정
yvar = 'Target'

# 훈련셋 특성 행렬 생성
X = train.drop(columns = yvar)

# 훈련셋 타겟 벡터 생성
y = train[yvar].copy()

In [36]:
# 시험셋 특성 행렬 생성
X_test = test.copy()

## 데이터 전처리

In [37]:
# UID / Failure Type 제거 및 Product ID 인덱스 설정
X = X.drop(columns=['UDI', 'Failure Type', 'Num ID']).set_index(keys='Product ID')
X_test = X_test.drop(columns=['UDI', 'Failure Type']).set_index(keys='Product ID')

In [38]:
X.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'],
      dtype='object')

In [39]:
X_test.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'],
      dtype='object')

In [41]:
# 컬럼명 변경
X.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear']
X_test.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear']

In [42]:
X.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
Product ID,,,,,,
M14860,M,298.1,308.6,1551,42.8,0
L47181,L,298.2,308.7,1408,46.3,3
L47182,L,298.1,308.5,1498,49.4,5
L47183,L,298.2,308.6,1433,39.5,7
L47184,L,298.2,308.7,1408,40.0,9


### 피처 엔지니어링
- temp_diff = Process temp - Air temp → 열 방출 능력 (Heat Dissipation Failure와 직결)
- power = Torque × Rotational speed → 기계 출력 (Power Failure와 직결)
- wear_torque = Tool wear × Torque → 마모 상태에서의 부하 (Overstrain Failure와 직결)
- Type 인코딩: L/M/H는 품질 등급 → OrdinalEncoder (L=0, M=1, H=2)

### 클래스 불균형 처리
- class_weight='balanced' 사용
- SMOTE 오버샘플링 고려
- 평가 메트릭: accuracy 대신 ROC-AUC 사용